# 09 · Searching for the fault geometry

Every chapter so far has assumed we already knew where the fault was: its
location, strike, dip, and size were fixed, and only the slip was unknown. That
assumption is what made the problem *linear*, letting us write
$\mathbf{d} = G\mathbf{m}$ and solve it in one step. This chapter relaxes it.
When the geometry itself is unknown, the problem becomes *nonlinear*, and we
have to search. The good news is that a clever trick lets us keep reusing the
fast linear solver from the earlier chapters inside the search.

**Learning objectives**

By the end of the chapter, you will be able to:

- explain why unknown geometry makes the inverse problem nonlinear
- use variable projection to keep slip linear while searching over geometry
- run a grid search over a geometry parameter and read its misfit curve
- refine the estimate with a bounded optimizer
- recognize a trade-off valley and know why it limits what geometry you can
  resolve

**Prerequisites:** Chapters 03, 04, and 08
**Estimated time:** about 60 minutes

GeoDef's axes, signs, and units are summarized in
[the conventions guide](../docs/conventions.md). New terms are introduced in
words here and collected in [the glossary](../docs/glossary.md).

## 1. Why unknown geometry is nonlinear

Recall what made slip linear: once the fault was fixed, each patch contributed
a fixed column to $G$, and the data were just a weighted sum of those columns.
Doubling a slip value doubled its contribution, exactly.

Geometry does not behave that way. If you change the fault's dip, you change
the shape of *every* column of $G$ at once, and not in proportion to anything
simple. There is no matrix that turns "dip" into "data" by multiplication. The
dependence is buried inside the Okada formulas,

$$ \mathbf{d} = G(\boldsymbol{\theta})\,\mathbf{m} + \boldsymbol{\varepsilon}, $$

where $\boldsymbol{\theta}$ collects the geometry parameters (position, strike,
dip, length, width). Because there is no closed-form solution for
$\boldsymbol{\theta}$, we have to try candidate geometries and see which one
fits: a **search**.

## 2. Variable projection: keep the easy part easy

Searching over geometry *and* every slip value at once would be a huge
nonlinear problem. The key insight avoids that. For *any* fixed trial geometry,
recovering the slip is still the ordinary linear inversion of Chapters 03 to
08. So we split the problem in two:

- an **inner** linear solve for slip, run at each trial geometry;
- an **outer** nonlinear search over just the handful of geometry parameters.

Concretely, define the misfit *after the best slip has been fitted*:

$$ \Phi(\boldsymbol{\theta}) = \min_{\mathbf{m}}
   \left[\, \lVert G(\boldsymbol{\theta})\mathbf{m} - \mathbf{d}\rVert^2_{C_d}
   + \lambda\lVert L\mathbf{m}\rVert^2 \,\right]. $$

For a candidate geometry, we solve the inner problem, read off how well it
fits, and hand that single number to the outer search. This strategy is called
**variable projection**. It shrinks a problem in hundreds of unknowns down to a
search in just a few, and it reuses machinery we already trust.

## 3. Rebuild the shared scenario

We use the same megathrust, true slip, GNSS network, and 1 cm noise from
Chapter 03. The one twist: we will *pretend not to know* the true dip of 15
degrees and try to recover it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

%matplotlib inline

rng = np.random.default_rng(0)

In [ ]:
# The shared megathrust, true slip, GNSS network, and 1 cm noise.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=315.0, dip=15.0,
    length=180e3, width=90e3,
    n_length=12, n_width=6,
)
N = fault.n_patches
n_length, n_width = fault.grid_shape
along = np.arange(N) % n_length
down = np.arange(N) // n_length
bump = np.exp(-(((along - (n_length - 1) / 2) / 3.0) ** 2
                + ((down - n_width / 2) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

grid_lon, grid_lat = np.meshgrid(np.linspace(98.5, 101.5, 8),
                                 np.linspace(-3.6, -0.4, 8))
station_lon, station_lat = grid_lon.ravel(), grid_lat.ravel()
n_stations = station_lon.size

G_full = fault.greens_matrix(station_lat, station_lon)
sigma = 0.01
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, 3 * n_stations)
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=d_obs[0::3], north=d_obs[1::3], up=d_obs[2::3],
    sigma_east=np.full(n_stations, sigma),
    sigma_north=np.full(n_stations, sigma),
    sigma_up=np.full(n_stations, sigma),
)
print(f"{N} patches, {n_stations} stations; true dip 15 degrees (pretend unknown)")

## 4. The reduced objective in code

Variable projection becomes two small functions. The first rebuilds the fault
at a trial dip (all other geometry parameters held at their true values). The
second runs the inner linear inversion there and returns its reduced
chi-squared, our single misfit number for that geometry.

In [ ]:
def make_fault(dip):
    # Rebuild the fault at a trial dip; everything else is held fixed.
    return geodef.Fault.planar(
        lat=-2.0, lon=100.0, depth=25e3, strike=315.0, dip=dip,
        length=180e3, width=90e3, n_length=12, n_width=6,
    )

def misfit(dip):
    # The inner linear solve: how well does the best slip at this dip fit?
    result = geodef.solve(make_fault(dip), gnss, components="dip",
                          regularization="laplacian", regularization_strength=1.0)
    return result.reduced_chi2

## 5. A grid search over dip

The simplest outer search just tries many dips and plots the misfit. This
**grid search** is slow but foolproof: it cannot get trapped, and it reveals
the whole shape of the objective, including any second minima that a local
optimizer might fall into.

In [ ]:
trial_dips = np.arange(5.0, 41.0, 2.5)
misfit_values = np.array([misfit(d) for d in trial_dips])
best_grid_dip = trial_dips[np.argmin(misfit_values)]
print(f"grid-search best dip: {best_grid_dip:g} degrees")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(trial_dips, misfit_values, "o-")
ax.axvline(15.0, color="green", ls="--", label="true dip (15)")
ax.axvline(best_grid_dip, color="red", ls=":", label=f"grid best ({best_grid_dip:g})")
ax.set(xlabel="trial dip (degrees)", ylabel="reduced chi-squared",
       title="Misfit versus fault dip")
ax.legend()
plt.show()

The misfit dips to a clear minimum near the true value. Notice that the curve
is not symmetric: assuming too steep a dip is penalized more sharply than
assuming too shallow a one. That asymmetry is a hint that dip trades off
against the other geometry parameters, a point we return to in Section 7.

## 6. Refine with a local optimizer

The grid told us roughly where the minimum is; a **local optimizer** polishes
it. Starting inside the basin we just found, a bounded one-dimensional
optimizer walks downhill to the precise minimum with far fewer evaluations than
a fine grid would need. For several geometry parameters at once you would hand a
vector to `scipy.optimize.minimize`; the idea is identical, and Chapter 10
makes it faster still with automatic derivatives.

In [ ]:
from scipy.optimize import minimize_scalar

optimum = minimize_scalar(misfit, bounds=(8.0, 25.0), method="bounded")
print(f"refined dip: {optimum.x:.2f} degrees  (true 15.0)")
print(f"reduced chi-squared at optimum: {optimum.fun:.3f}")

## 7. The recovered geometry and slip, and the trade-off

At the recovered dip, the inner inversion returns a sensible bump: geometry and
slip together explain the data. The recovered dip is close to the truth but not
exact, and the reason is important.

In [ ]:
best_fault = make_fault(optimum.x)
best = geodef.solve(best_fault, gnss, components="dip",
                    regularization="laplacian", regularization_strength=1.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
vmax = slip_true[N:].max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title="True (dip 15)",
                 colorbar_label="Slip (m)")
geodef.plot.slip(best_fault, best.dip_slip, ax=ax2, cmap="RdBu_r",
                 vmin=-vmax, vmax=vmax, title=f"Recovered (dip {optimum.x:.1f})",
                 colorbar_label="Slip (m)")
plt.tight_layout()
plt.show()

The reason the recovered dip is not pinned to exactly 15 degrees is a
**trade-off**. A slightly wrong dip can be partly compensated by a slightly
different slip distribution, and the two changes together still fit the noisy
data almost as well. When misfit barely changes along a direction in parameter
space, that direction is poorly resolved: the data simply do not distinguish
those models. A wide, flat misfit valley is the geometry-search version of the
ill-conditioning we met in Chapter 03.

This is exactly why a grid search matters. A single optimizer result reports a
best dip but hides the *shape* around it. Plotting the objective, as in
Section 5, is what reveals whether that best value is sharply determined or
sits in a long flat valley.

**Checkpoint.** The grid search evaluated `misfit` at each trial dip, and each
call ran a full linear inversion. If we searched over three geometry parameters
on a grid of 15 values each, how many inversions would that be? What does the
number suggest about grid search in higher dimensions?

<details><summary>Answer</summary>

$15^3 = 3375$ inversions, and a fourth parameter would push it past fifty
thousand. Grid search is safe but its cost explodes with the number of
parameters, which is why we use it to map one or two dimensions and then switch
to a local optimizer (Chapter 09) or gradient-based search (Chapter 10) for the
rest.

</details>

## Checkpoint questions

**Why solve for slip inside each geometry trial, rather than searching over
slip and geometry together?**

<details><summary>Answer</summary>

Slip is linear given the geometry, so its best value can be computed exactly and
cheaply with the solver we already have. Folding it into the nonlinear search
would turn a few-parameter search into a several-hundred-parameter one, for no
benefit. Variable projection keeps the easy part easy.

</details>

**What does a long, flat misfit valley tell you?**

<details><summary>Answer</summary>

That the geometries along the valley are not separately identifiable from these
data: different combinations fit about equally well because slip absorbs the
difference. The best-fit point in such a valley is poorly constrained, and a
reported error bar from a local optimizer will understate the true ambiguity.

</details>

**Why run a grid search before a local optimizer?**

<details><summary>Answer</summary>

A local optimizer only walks downhill from where it starts, so it can settle
into whichever basin it happens to begin in. The grid reveals how many basins
exist and where the boundaries lie, so you can start the optimizer in the right
one and know whether the minimum is unique.

</details>

## Common mistakes

- **Reporting only that the optimizer succeeded.** Success means it found *a*
  minimum, not *the* minimum. Always show the objective map or several restarts
  so the reader can judge whether the basin is unique.
- **Changing regularization between geometries.** If you vary the smoothing
  strength while searching over dip, you are changing two things at once and
  cannot attribute the misfit change to geometry alone. Hold the prior fixed.
- **Trusting a local error bar on a multimodal surface.** A curvature-based
  uncertainty describes one basin. Across a surface with several minima or a
  long valley, it is meaningless.

## Recap

- Unknown geometry makes the forward model nonlinear, because geometry reshapes
  every column of $G$ at once.
- Variable projection keeps slip linear (an inner solve) and searches only over
  the few geometry parameters (an outer loop).
- A grid search maps the objective safely; a local optimizer refines the
  minimum cheaply.
- Trade-off valleys mark geometry directions the data cannot resolve, and only
  a map of the objective reveals them.

Chapter 10 speeds this search up dramatically using automatic differentiation
to compute exact gradients of the objective.

## Exercises

Predict the qualitative outcome before running each experiment. Worked
solutions are in `solutions/09_nonlinear_geometry_solutions.ipynb`.

1. **Scan and restart.** Scan dip, then refine the optimizer from starting
   points on both sides of the minimum. Do they converge to the same value?
2. **A bad basin.** Start the optimizer from a deliberately poor initial dip
   (say 40 degrees) and document where it ends up and why.
3. **Noisier data.** Double the noise level, repeat the dip scan, and describe
   how the minimum sharpens or broadens.
4. **Challenge: a trade-off map.** Search over depth and width together, plot
   the two-dimensional misfit surface, and explain the trade-off valley
   physically (why can a deeper, larger source mimic a shallower, smaller one?).

## Further reading

- Golub, G., and Pereyra, V. (1973), "The differentiation of pseudo-inverses
  and nonlinear least squares problems whose variables separate," *SIAM Journal
  on Numerical Analysis*, 10(2), 413–432, the origin of variable projection.
- Tarantola, A. (2005), *Inverse Problem Theory*, on nonlinear inverse
  problems and objective surfaces.
- Segall, P. (2010), *Earthquake and Volcano Deformation*, on geometry and slip
  trade-offs.
- [GeoDef inversion documentation](../docs/invert.md) for the geometry-search
  interface used in the next chapter.
- The next course chapter is `10_gradient_geometry.ipynb`.